In [ ]:
import liesel.goose as gs
import liesel.optim as loptim
import optax

import ppptm as gptm

In [ ]:
def opt1():
    # Define the learning rate scheduler
    scheduler = optax.cosine_decay_schedule(init_value=0.01, decay_steps=2000)

    # Define the AdamW optimizer with weight decay
    adamw_optimizer = optax.adamw(learning_rate=scheduler, weight_decay=1e-4)

    gradient_clipping = optax.clip_by_global_norm(1.0)

    # Combine them using optax.chain
    combined_optimizer = optax.chain(gradient_clipping, adamw_optimizer)
    return combined_optimizer

In [ ]:
data = gptm.load_americas()

In [ ]:
locs = gptm.LocationVars.new_from(data.locs, n_subset=256)
y = data.obs[:, locs.locs.ordering]
model = gptm.Model.new_HG(
    y,
    locs,
    g_dist=gptm.G(y, locs, ard=True).new_gaussian(),
    coef=gptm.H(locs, ard=True).new_coef(),
)

In [ ]:
result = model.fit(
    loss_monitor=loptim.EmaTrainLossMonitor(effective_window=1.0),
    stopper=gs.Stopper(max_iter=2_000, patience=30),
    optimizer=opt1(),
    progress_bar=True,
)
position = result.history.position
assert position is not None

In [ ]:
result.plot_loss()

In [ ]:
result.plot_params(subset=[k for k in position if "amplitude" in k])

In [ ]:
result.plot_params(subset=[k for k in position if "length_scale" in k])

In [ ]:
result.plot_params(subset=[k for k in position if "loc_mean" in k])

In [ ]:
result.plot_params(subset=[k for k in position if "scale_mean" in k])

In [ ]:
df = gptm.long_df_multiple(
    locs.locs.ordered,
    mu=model.graph.vars["loc"].value,
    sigma=model.graph.vars["scale"].value,
)

In [ ]:
gptm.plot_df(df, "mu")

In [ ]:
gptm.plot_df(df, "sigma")

In [ ]:
model.coef.value.shape

In [ ]:
df = gptm.long_df(locs.locs.ordered, model.coef.value.T)
gptm.plot_df(df.query("obs == 20"))

In [ ]:
gptm.plot_df(df.query("obs == 30"))

In [ ]:
model.spatial_coef.amplitude.value